In [ ]:
from codecs import charmap_build
from typing_extensions import no_type_check
import RPi.GPIO as GPIO
import time
import random
import string


button = 17 # change this based on wiring

GPIO.setmode(GPIO.BCM)
GPIO.setup(button, GPIO.IN, pull_up_down=GPIO.PUD_UP)

morse_dict = {".-":"A","-...":"B", "-.-.":"C", "-..":"D", ".":"E", "..-.":"F", "--.":"G", "....":"H", "..":"I", ".---":"J", "-.-":"K", ".-..":"L", "--":"M", "-.":"N", "---":"O", ".--.":"P", "--.-":"Q", ".-.":"R", "...":"S", "-":"T", "..-":"U", "...-":"V", ".--":"W", "-..-":"X", "-.--":"Y", "--..":"Z",}
historical_rotors = {'I':   {'wiring': 'EKMFLGDQVZNTOWYHXUSPAIBRCJ', 'notch': 'Q'},'II':  {'wiring': 'AJDKSIRUXBLHWTMCQGZNPYFVOE', 'notch': 'E'},'III': {'wiring': 'BDFHJLCPRTXVZNYEIWGAKMUSQO', 'notch': 'V'},'IV':  {'wiring': 'ESOVPZJAYQUIRHXLNFTGKDCMWB', 'notch': 'J'},'V':   {'wiring': 'VZBRGITYUPSDNHLXAWMJQOFECK', 'notch': 'Z'},}
reflector_b = "YRUHQSLDPXNGOKMIEBFZCWVJAT"

dot_threshold = 0.3
letter_pause = 0.8

class Rotor:
  def __init__(self, wiring, notch, start_character="A"):
    self.alphabet = string.ascii_uppercase
    self.wiring = wiring
    self.notch = notch
    self.position = self.alphabet.index(start_character)

  def rotate(self) -> bool:
    at_notch = (self.alphabet[self.position] == self.notch)
    self.position = (self.position + 1) % 26
    return at_notch

  def forward(self,character_name: int) -> int:
    shift_ts = (character_name + self.position) % 26
    wired_character = self.wiring[shift_ts]
    out_put = (self.alphabet.index(wired_character)-self.position) % 26
    return out_put

  def backward(self, character_name: int) -> int:
    shift_ts = (character_name + self.position) % 26
    shift_character = self.alphabet[shift_ts]
    wired_part = self.wiring.index(shift_character)
    out_put = (wired_part - self.position) % 26
    return out_put

def create_randomized_rotors():
  selected_names = random.sample(list(historical_rotors.keys()), 3)
  start_positions = [random.choice(string.ascii_uppercase) for _ in range(3)]

  rotors = []
  for name, start_position in zip(selected_names, start_positions):
    data = historical_rotors[name]
    rotors.append(Rotor(data["wiring"], data["notch"], start_position))

  print(f"selected rotors: {selected_names}")
  print(f"initial positions: {start_positions}")

  return rotors[0], rotors[1], rotors[2]

def step_rotors(left_rotor, middle_rotor, right_rotor):
  right_at_notch = (right_rotor.alphabet[right_rotor.position] == right_rotor.notch)
  middle_at_notch = (middle_rotor.alphabet[middle_rotor.position] == middle_rotor.notch)

  if middle_at_notch:
    middle_rotor.rotate()
    left_rotor.rotate()
  elif right_at_notch:
    middle_rotor.rotate()

  right_rotor.rotate()


def encode_character(char: str, left: Rotor, middle: Rotor, right: Rotor) -> str:
  if char not in string.ascii_uppercase:
    return char

  step_rotors(left, middle, right)

  triple_t = string.ascii_uppercase.index(char)

  triple_t = right.forward(triple_t)
  triple_t = middle.forward(triple_t)
  triple_t = left.forward(triple_t)

  reflector_character = reflector_b[triple_t]
  triple_t = string.ascii_uppercase.index(reflector_character)

  triple_t = left.backward(triple_t)
  triple_t = middle.backward(triple_t)
  triple_t = right.backward(triple_t)

  return string.ascii_uppercase[triple_t]

left_r, middle_r, right_r = create_randomized_rotors()

current_symbol = ""
pressed_time = None
last_event_time = time.time()

print("Start entering Morse code")

try:
  while True:
    if GPIO.input(button) == GPIO.LOW:
      if pressed_time is None:
        pressed_time = time.time()
    else:
      if pressed_time is not None:
        pressed_length = time.time() - pressed_time
        pressed_time = None

        if pressed_length < dot_threshold:
          current_symbol += "."
          print(".", end="", flush= True)
        else:
          current_symbol += "-"
          print("-", end="", flush=True)

        last_event_time = time.time()

    pause = time.time() - last_event_time


    if current_symbol and pause > letter_pause:
      letter = morse_dict.get(current_symbol,"?" )
      cipher_text = encode_character(letter, left_r, middle_r, right_r)
      print(f"code: {cipher_text}")
      current_symbol = ""
      last_event_time = time.time()

    time.sleep(0.01)

except KeyboardInterrupt:
  GPIO.cleanup()